System information (for reproducibility):

In [39]:
versioninfo()

Julia Version 1.12.6
Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 12 × Apple M2 Max
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m2)
  GC: Built with stock GC
Threads: 8 default, 1 interactive, 8 GC (on 8 virtual cores)
Environment:
  JULIA_NUM_THREADS = 8
  JULIA_EDITOR = code


Load packages:

In [40]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Documents/github.com/ucla-biostat-257/2026spring/slides/19-easylineq`


Status `~/Documents/github.com/ucla-biostat-257/2026spring/slides/19-easylineq/Project.toml`
  [6e4b80f9] BenchmarkTools v1.8.0
  [42fd0dbc] IterativeSolvers v0.9.4
  [7ed4a6bd] LinearSolve v3.75.0
  [b51810bb] MatrixDepot v1.0.15
  [af69fa37] Preconditioners v0.6.2
  [b8865327] UnicodePlots v3.8.2
  [efce3f68] WoodburyMatrices v1.1.0
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0
  [2f01184e] SparseArrays v1.12.0


# Introduction

Consider $\mathbf{A} \mathbf{x} = \mathbf{b}$, $\mathbf{A} \in \mathbb{R}^{n \times n}$. Or, consider matrix inverse (if you want). $\mathbf{A}$ can be huge. Keep massive data in mind: 1000 Genome Project, NetFlix, Google PageRank, finance, spatial statistics, ... We should be alert to many easy linear systems. 

Don't blindly use `A \ b` and `inv` in Julia or `solve` function in R. **Don't waste computing resources by bad choices of algorithms!**

## Diagonal matrix

Diagonal $\mathbf{A}$: $n$ flops. Use `Diagonal` type of Julia.

In [41]:
using BenchmarkTools, LinearAlgebra, Random

# generate random data
Random.seed!(257)
n = 1000
A = diagm(0 => randn(n)) # a diagonal matrix stored as Matrix{Float64}
b = randn(n);

In [42]:
# should give link to the source code
@which A \ b

\(A::AbstractMatrix, B::AbstractVecOrMat)
     @ LinearAlgebra ~/.julia/juliaup/julia-1.12.6+0.aarch64.apple.darwin14/share/julia/stdlib/v1.12/LinearAlgebra/src/generic.jl:1220

In [43]:
# check `istril(A)` and `istriu(A)` (O(n^2)), then call `Diagonal(A) \ b` (O(n))
@benchmark $A \ $b

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  110.458 μs …  4.254 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     120.000 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   122.082 μs ± 48.231 μs  ┊ GC (mean ± σ):  0.20% ± 0.95%

                 ▇█▆▆▄▃▂▂▁▁ ▁                                  ▂
  ▄▄▁▁▁▃▁▁▁▁▁▁▁▄▆█████████████▇███████▇▇▇▇▇▆▆▆▆▆▆▆▇▆▆▆▆▇▆▆▅▅▆▆ █
  110 μs        Histogram: log(frequency) by time       145 μs <

 Memory estimate: 16.12 KiB, allocs estimate: 6.

In [44]:
# O(n) computation, no extra array allocation
@benchmark Diagonal($A) \ $b

BenchmarkTools.Trial: 10000 samples with 16 evaluations per sample.
 Range (min … max):  955.688 ns … 86.766 μs  ┊ GC (min … max):  0.00% … 95.87%
 Time  (median):       1.062 μs              ┊ GC (median):     0.00%
 Time  (mean ± σ):     1.324 μs ±  3.916 μs  ┊ GC (mean ± σ):  17.17% ±  5.74%

   ▁ ▁ ▂▆▆█▁                                                    
  ▃█▇███████▆▃▂▂▁▁▁▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  956 ns          Histogram: frequency by time         1.83 μs <

 Memory estimate: 16.12 KiB, allocs estimate: 6.

## Bidiagonal, tridiagonal, and banded matrices

Bidiagonal, tridiagonal, or banded $\mathbf{A}$: Band LU, band Cholesky, ... roughly $O(n)$ flops.   
* Use [`Bidiagonal`](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.Bidiagonal), [`Tridiagonal`](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.Tridiagonal), [`SymTridiagonal`](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.SymTridiagonal) types of Julia.

In [45]:
Random.seed!(257) 

n  = 1000
dv = randn(n)
ev = randn(n - 1)
b  = randn(n) # rhs
# symmetric tridiagonal matrix
A  = SymTridiagonal(dv, ev)

1000×1000 SymTridiagonal{Float64, Vector{Float64}}:
 0.679063   0.817275    ⋅        …    ⋅          ⋅          ⋅ 
 0.817275   1.24568   -0.527485       ⋅          ⋅          ⋅ 
  ⋅        -0.527485  -1.21007        ⋅          ⋅          ⋅ 
  ⋅          ⋅         0.187263       ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅        …    ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
 ⋮                               ⋱                        
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅          ⋅ 
  ⋅    

In [46]:
# convert to a full matrix
Afull = Matrix(A)

# LU decomposition (2/3) n^3 flops!
@benchmark $Afull \ $b

BenchmarkTools.Trial: 1097 samples with 1 evaluation per sample.
 Range (min … max):  4.177 ms …  15.719 ms  ┊ GC (min … max): 0.00% … 10.82%
 Time  (median):     4.347 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.558 ms ± 665.016 μs  ┊ GC (mean ± σ):  2.01% ±  4.40%

  ▅▇█▅▄▄▄▃▁  ▁▃▃▁ ▁                                            
  ██████████▆███████▇▇▄▄▁▅▁▅▅▅▄▅▅▁▁▁▁▄▄▁▁▄▄▁▁▁▁▅▄▁▄▄▄▁▄▁▁▁▁▁▄ █
  4.18 ms      Histogram: log(frequency) by time      7.63 ms <

 Memory estimate: 7.66 MiB, allocs estimate: 9.

In [47]:
# specialized algorithm for tridiagonal matrix, O(n) flops
@benchmark $A \ $b

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  12.583 μs … 922.875 μs  ┊ GC (min … max): 0.00% … 96.43%
 Time  (median):     13.834 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   14.725 μs ±  21.932 μs  ┊ GC (mean ± σ):  4.07% ±  2.70%

            █▇▆▆▅▄▅▄▃▁▁                                        ▂
  ▆▅▅▆▆▇▆▅▄▅█████████████▇▆▅▆▅▅▅▆▅▅▅▅▅▅▅▅▅▅▅▄▆▆▄▆▅▆▇▇▆▇▆▃▆▃▅▄▅ █
  12.6 μs       Histogram: log(frequency) by time      18.8 μs <

 Memory estimate: 24.19 KiB, allocs estimate: 9.

## Triangular matrix

Triangular $\mathbf{A}$: $n^2$ flops to solve linear system.

In [48]:
Random.seed!(257)

n = 1000
A = tril(randn(n, n)) # a lower-triangular matrix stored as Matrix{Float64}
b = randn(n)

# check istril() then triangular solve
@benchmark $A \ $b

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  152.125 μs … 381.708 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     154.083 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   180.648 μs ±  56.124 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▅▄▃▃▂▂▁▁▁                                     ▁▂▃▁▂▁▂▃▁      ▁
  ████████████▇▇▇▇▇▆▅▄▅▁▅▅▅▅▃▄▄▄▁▄▁▁▃▁▁▁▁▄▁▁▁▁▁▁█████████████▇█ █
  152 μs        Histogram: log(frequency) by time        335 μs <

 Memory estimate: 8.06 KiB, allocs estimate: 3.

In [49]:
# triangular solve directly; save the cost of istril()
@benchmark LowerTriangular($A) \ $b

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  86.167 μs …  1.833 ms  ┊ GC (min … max): 0.00% … 90.48%
 Time  (median):     93.583 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   95.454 μs ± 18.716 μs  ┊ GC (mean ± σ):  0.17% ±  0.90%

                 █▇▃▂▃▁ ▂▄▂▂     ▁▁                           ▂
  ▄▁▁▁▃▁▁▃▁▁▁▃▁▁▁█████████████▇███████▇▆▇███▇▇▆▇▇▇▇▇█▇▆▆▇▆▇▇▇ █
  86.2 μs      Histogram: log(frequency) by time       113 μs <

 Memory estimate: 8.06 KiB, allocs estimate: 3.

## Block diagonal matrix

Block diagonal: Suppose $n = \sum_b n_b$. For linear equations, $(\sum_b n_b)^3$ (without using block diagonal structure) vs $\sum_b n_b^3$ (using block diagonal structure).  

Julia has a [`blockdiag`](https://docs.julialang.org/en/v1/stdlib/SparseArrays/#SparseArrays.blockdiag) function that generates a **sparse** matrix. **Anyone interested writing a `BlockDiagonal.jl` package?**

In [50]:
using SparseArrays

Random.seed!(257)

B  = 10 # number of blocks
ni = 100
A  = blockdiag([sprandn(ni, ni, 0.01) for b in 1:B]...)

1000×1000 SparseMatrixCSC{Float64, Int64} with 998 stored entries:
⎡⡨⣶⣿⢼⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⢨⡿⣾⣞⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠉⠉⠁⢿⡺⣽⣶⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⣻⣻⠯⣹⠄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠈⠀⠉⠁⣾⣼⣜⣏⠂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⣿⡷⣭⠞⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠀⣲⡩⢿⣞⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣯⣲⠽⢮⠄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⠀⠁⢟⣽⢟⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢿⣿⡿⣭⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣾⣲⣾⣽⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣫⣿⣟⣿⠀⢀⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⣷⣺⣟⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠐⣵⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠐⣿⣻⣬⣟⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡧⣱⢾⣎⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠨⣫⣗⣱⡿⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠐⣾⣞⡿⡑⣀⠀⣀⢀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢘⣿⣳⣟⣉⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡟⣿⢝⠟⎦

In [51]:
using UnicodePlots
spy(A)

         ┌──────────────────────────────┐    
       1 │⣿⣿⣛⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ > 0
         │⠛⠓⠻⡤⣶⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ < 0
         │⠀⠀⠀⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠈⣾⢿⣯⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠛⠙⠫⢄⣴⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠈⣿⡾⣿⣀⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣟⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠛⠛⣤⣠⢤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣻⣿⣿⠀⡀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⠛⠛⣤⢤⣤⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣟⣯⣟⡀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣯⣿⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⠛⢫⣤⣤⡴│    
   1 000 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⡿│    
         └──────────────────────────────┘    
         ⠀1⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀1 000⠀    
         ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀998 ≠ 0⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀    

## Kronecker product

Use
$$
\begin{eqnarray*}
    (\mathbf{A} \otimes \mathbf{B})^{-1} &=& \mathbf{A}^{-1} \otimes \mathbf{B}^{-1} \\
    (\mathbf{C}^T \otimes \mathbf{A}) \text{vec}(\mathbf{B}) &=& \text{vec}(\mathbf{A} \mathbf{B} \mathbf{C}) \\
    \text{det}(\mathbf{A} \otimes \mathbf{B}) &=& [\text{det}(\mathbf{A})]^p [\text{det}(\mathbf{B})]^m, \quad \mathbf{A} \in \mathbb{R}^{m \times m}, \mathbf{B} \in \mathbb{R}^{p \times p}
\end{eqnarray*}    
$$
to avoid forming and doing costly computation on the potentially huge Kronecker $\mathbf{A} \otimes \mathbf{B}$.

**Anyone interested writing a package?**

In [52]:
using MatrixDepot, LinearAlgebra

A = matrixdepot("lehmer", 50) # a pd matrix

50×50 Matrix{Float64}:
 1.0        0.5        0.333333   0.25       …  0.0208333  0.0204082  0.02
 0.5        1.0        0.666667   0.5           0.0416667  0.0408163  0.04
 0.333333   0.666667   1.0        0.75          0.0625     0.0612245  0.06
 0.25       0.5        0.75       1.0           0.0833333  0.0816327  0.08
 0.2        0.4        0.6        0.8           0.104167   0.102041   0.1
 0.166667   0.333333   0.5        0.666667   …  0.125      0.122449   0.12
 0.142857   0.285714   0.428571   0.571429      0.145833   0.142857   0.14
 0.125      0.25       0.375      0.5           0.166667   0.163265   0.16
 0.111111   0.222222   0.333333   0.444444      0.1875     0.183673   0.18
 0.1        0.2        0.3        0.4           0.208333   0.204082   0.2
 ⋮                                           ⋱                        
 0.0238095  0.047619   0.0714286  0.0952381     0.875      0.857143   0.84
 0.0232558  0.0465116  0.0697674  0.0930233     0.895833   0.877551   0.86
 0.02272

In [53]:
B = matrixdepot("oscillate", 100) # pd matrix

100×100 Matrix{Float64}:
  0.707518      0.0578848    -0.00202508   …  -2.07618e-11   3.97326e-12
  0.0578848     0.433525      0.0956315        1.36333e-10  -2.60905e-11
 -0.00202508    0.0956315     0.807263        -1.45959e-10   2.79326e-11
  0.00464379   -0.0293052     0.0908958        8.28755e-10  -1.58602e-10
 -0.000796805   0.00596484   -0.00873108      -5.8795e-10    1.12518e-10
  0.000152015  -0.00249669    0.00519362   …   8.62146e-10  -1.64992e-10
  1.24633e-5    0.000187569  -0.00288517      -1.75966e-9    3.36752e-10
 -3.61382e-5   -3.52123e-5    0.000736403      1.38969e-9   -2.65949e-10
 -0.000252717   0.00129378   -0.000357227     -2.23571e-9    4.27855e-10
  2.10061e-5   -0.000141756   0.000188028      9.45806e-10  -1.81002e-10
  ⋮                                        ⋱                
  2.41765e-11  -1.58755e-10   1.69964e-10      0.000337715  -6.45464e-5
 -3.38657e-11   2.2238e-10   -2.38081e-10     -0.000474301   9.05451e-5
  4.06352e-12  -2.66832e-11   2.85671e-1

In [54]:
M = kron(A, B)
c = ones(size(M, 2)) # rhs
# Method 1: form Kronecker product and Cholesky solve
x1 = cholesky(Symmetric(M)) \ c;

In [55]:
# Method 2: use (A ⊗ B)^{-1} = A^{-1} ⊗ B^{-1}
m, p = size(A, 1), size(B, 1)
x2 = vec(transpose(cholesky(Symmetric(A)) \ 
    transpose(cholesky(Symmetric(B)) \ reshape(c, p, m))));

In [56]:
# relative error
norm(x1 - x2) / norm(x1)

9.928565905232742e-8

In [57]:
using BenchmarkTools

# Method 1: form Kronecker and Cholesky solve
@benchmark cholesky(Symmetric(kron($A, $B))) \ c

BenchmarkTools.Trial: 27 samples with 1 evaluation per sample.
 Range (min … max):  158.714 ms … 241.023 ms  ┊ GC (min … max):  7.94% … 28.30%
 Time  (median):     171.845 ms               ┊ GC (median):     9.61%
 Time  (mean ± σ):   191.154 ms ±  29.372 ms  ┊ GC (mean ± σ):  21.06% ± 11.68%

  ▃█  █    ▃                             ▃    █                  
  ██▇▇█▇▁▇▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇█▇▇▁▁█▇▇▁▁▁▁▇▁▁▁▇▁▁▁▁▇ ▁
  159 ms           Histogram: frequency by time          241 ms <

 Memory estimate: 381.55 MiB, allocs estimate: 10.

In [58]:
# Method 2: use (A ⊗ B)^{-1} = A^{-1} ⊗ B^{-1}
@benchmark vec(transpose(cholesky(Symmetric($A)) \ 
    transpose(cholesky(Symmetric($B)) \ reshape($c, p, m))))

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  62.750 μs …  4.368 ms  ┊ GC (min … max): 0.00% … 98.23%
 Time  (median):     66.208 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   71.269 μs ± 59.377 μs  ┊ GC (mean ± σ):  5.10% ±  7.08%

   ██▆                                                         
  ▃███▆▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  62.8 μs         Histogram: frequency by time         138 μs <

 Memory estimate: 196.50 KiB, allocs estimate: 18.

## Sparse matrix

Sparsity: sparse matrix decomposition or iterative method.  

* The easiest recognizable structure. Familiarize yourself with the sparse matrix computation tools in Julia, Matlab, R (`Matrix` package), MKL (sparse BLAS), ... as much as possible.

In [59]:
using MatrixDepot

Random.seed!(257)

# a 7701-by-7701 sparse pd matrix
A = matrixdepot("wathen", 50)
# random generated rhs
b = randn(size(A, 1))
Afull = Matrix(A)
count(!iszero, A) / length(A) # sparsity

0.001994776158751544

In [60]:
using UnicodePlots
spy(A)

         ┌──────────────────────────────┐    
       1 │⢻⣶⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ > 0
         │⠀⠙⢿⣷⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ < 0
         │⠀⠀⠀⠘⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠙⠻⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠿⣧⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢻⣶⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠿⣧⣄⠀⠀⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⡄⠀⠀⠀│    
         │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢿⣷⣄⠀│    
   7 701 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠿⣧│    
         └──────────────────────────────┘    
         ⠀1⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀7 701⠀    
         ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀118 301 ≠ 0⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀    

### Matrix-vector multiplication

In [61]:
# dense matrix-vector multiplication
@benchmark $Afull * $b

BenchmarkTools.Trial: 694 samples with 1 evaluation per sample.
 Range (min … max):  5.365 ms …  10.787 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     7.209 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   7.210 ms ± 690.368 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

                             ▂ ▁▂▁▃▇▅▄      ▂▇▄▅▃█▁            
  ▅▂▃▁▂▃▃▄▄▃▃▅▅▅▅▆▃▂▃▃▁▂▅▃▄▄▅█████████▇▇▃▅▅████████▆█▄▄▄▃▃▃▁▃ ▄
  5.36 ms         Histogram: frequency by time        8.43 ms <

 Memory estimate: 64.06 KiB, allocs estimate: 3.

In [62]:
# sparse matrix-vector multiplication
@benchmark $A * $b

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  62.416 μs …  1.917 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     64.416 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   69.537 μs ± 41.813 μs  ┊ GC (mean ± σ):  1.47% ± 3.19%

  ▆█▆▄▅▄▃▃▂▂▂▂▂▂▁▁▁▂▂▁                                        ▂
  ████████████████████▇▇▇▆▇▆▅▅▆▆▇▇▆▅▅▆▅▄▅▄▅▅▃▃▄▅▄▄▅▃▁▄▄▃▃▃▃▄▃ █
  62.4 μs      Histogram: log(frequency) by time       126 μs <

 Memory estimate: 64.06 KiB, allocs estimate: 3.

### Solve linear equation

In [63]:
# solve via dense Cholesky
xchol = cholesky(Symmetric(Afull)) \ b
@benchmark cholesky($(Symmetric(Afull))) \ $b

BenchmarkTools.Trial: 10 samples with 1 evaluation per sample.
 Range (min … max):  491.464 ms … 553.952 ms  ┊ GC (min … max): 0.14% … 0.12%
 Time  (median):     502.588 ms               ┊ GC (median):    0.13%
 Time  (mean ± σ):   511.548 ms ±  22.378 ms  ┊ GC (mean ± σ):  0.13% ± 0.01%

  █  ███ █      ██              █                       █     █  
  █▁▁███▁█▁▁▁▁▁▁██▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁█ ▁
  491 ms           Histogram: frequency by time          554 ms <

 Memory estimate: 452.53 MiB, allocs estimate: 6.

In [64]:
# solve via sparse Cholesky
xcholsp = cholesky(Symmetric(A)) \ b
@show norm(xchol - xcholsp)
@benchmark cholesky($(Symmetric(A))) \ $b

norm(xchol - xcholsp) = 3.725918455304583e-15


BenchmarkTools.Trial: 841 samples with 1 evaluation per sample.
 Range (min … max):  5.534 ms …  14.536 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     5.710 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   5.948 ms ± 661.419 μs  ┊ GC (mean ± σ):  2.41% ± 4.03%

  ▅█▆▄▃▃▂▁  ▂▄▃▂ ▁                                             
  ████████▆█████████▇▆▆▅▅▅▄▄▅▅▁▁▁▅▄▄▁▁▄▁▁▄▁▁▄▄▁▁▁▁▁▁▁▁▄▁▄▅▁▁▄ █
  5.53 ms      Histogram: log(frequency) by time         9 ms <

 Memory estimate: 12.55 MiB, allocs estimate: 65.

In [65]:
# sparse solve via conjugate gradient
using IterativeSolvers

xcg = cg(A, b)
@show norm(xcg - xchol)
@benchmark cg($A, $b)

norm(xcg - xchol) = 2.5708139737090377e-7


BenchmarkTools.Trial: 299 samples with 1 evaluation per sample.
 Range (min … max):  15.949 ms …  23.015 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     16.434 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   16.761 ms ± 948.486 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▆▅███▄▅▃▂                                                   
  ▆▆██████████▆▃▃▄▂▃▂▁▂▄▂▂▂▂▂▄▃▃▁▂▃▂▂▂▅▁▂▁▂▂▂▄▃▂▁▂▁▁▁▃▂▁▁▂▁▁▁▄ ▃
  15.9 ms         Histogram: frequency by time           20 ms <

 Memory estimate: 256.89 KiB, allocs estimate: 20.

In [66]:
# sparse solve via preconditioned conjugate gradient
using Preconditioners

xpcg = cg(A, b, Pl = DiagonalPreconditioner(A))
@show norm(xpcg - xchol)
@benchmark cg($A, $b, Pl = $(DiagonalPreconditioner(A)))

norm(xpcg - xchol) = 3.5729405649199994e-8


BenchmarkTools.Trial: 1494 samples with 1 evaluation per sample.
 Range (min … max):  3.198 ms …   4.753 ms  ┊ GC (min … max): 0.00% … 26.19%
 Time  (median):     3.304 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.348 ms ± 161.157 μs  ┊ GC (mean ± σ):  0.05% ±  0.92%

   █▆▄                                                         
  ▅██████▇▇▇▆▅▅▆▅▇▄▄▅▅▅▅▅▅▄▄▄▄▃▃▃▃▄▃▃▃▃▂▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▂▁▂▂▁▂ ▃
  3.2 ms          Histogram: frequency by time        3.86 ms <

 Memory estimate: 256.89 KiB, allocs estimate: 20.

## Easy plus low rank

Easy plus low rank: $\mathbf{U} \in \mathbb{R}^{n \times r}$, $\mathbf{V} \in \mathbb{R}^{r \times n}$, $r \ll n$. Woodbury formula
\begin{eqnarray*}
	(\mathbf{A} + \mathbf{U} \mathbf{V}^T)^{-1} &=& \mathbf{A}^{-1} - \mathbf{A}^{-1} \mathbf{U} (\mathbf{I}_r + \mathbf{V} \mathbf{A}^{-1} \mathbf{U}^T)^{-1} \mathbf{V}^T \mathbf{A}^{-1} \\
    \text{det}(\mathbf{A} + \mathbf{U} \mathbf{V}^T) &=& \text{det}(\mathbf{A}) \text{det}(\mathbf{I}_r + \mathbf{V} \mathbf{A}^{-1} \mathbf{U}^T).
\end{eqnarray*}

* Keep HW3 (multivariate density) and HW4 (PageRank) problems in mind.  

* [`WoodburyMatrices.jl`](https://github.com/timholy/WoodburyMatrices.jl) package can be useful.

In [67]:
using BenchmarkTools, Random, WoodburyMatrices

Random.seed!(257)
n = 1000
r = 5

A = Diagonal(rand(n))
B = randn(n, r)
D = Diagonal(rand(r))
b = randn(n)
# Woodbury structure: W = A + B * D * B'
W = SymWoodbury(A, B, D)
Wfull = Matrix(W) # stored as a Matrix{Float64}

1000×1000 Matrix{Float64}:
  1.59654    0.456106    1.52748   …  -0.725063   1.23432   -0.467818
  0.456106   3.23966     0.960678      1.22398   -0.256551  -0.191276
  1.52748    0.960678    3.39075       0.432445   1.34648   -1.06645
 -0.486356  -0.0292724  -1.36154      -0.998257  -0.378862   0.63341
 -1.12244    0.777027   -1.45956       1.42378   -1.84637    0.509516
  0.519072   1.93077     1.4578    …   1.88547   -0.250131  -0.539974
 -1.31967   -0.941968   -1.55701       1.24036   -1.73289    0.33383
 -0.365699   1.79832    -0.749824      1.03221   -0.893575   0.393119
 -0.206288  -1.30162    -0.764563     -1.75899    0.249943   0.334384
  0.598281   1.96579     2.04488       2.91997   -0.423147  -0.897774
  ⋮                                ⋱                        
  1.32891    1.73318     1.88389      -0.454571   1.60264   -0.38926
 -0.855297   2.71166    -0.811348      2.49806   -2.44808    0.339342
  0.101575   2.2306     -0.13939       0.201285  -0.590535   0.288461
 -1.03

In [68]:
# compares storage
Base.summarysize(W), Base.summarysize(Wfull)

(48480, 8000048)

### Solve linear equation

In [69]:
# solve via Cholesky
@benchmark cholesky($(Symmetric(Wfull))) \ $b

BenchmarkTools.Trial: 1598 samples with 1 evaluation per sample.
 Range (min … max):  2.490 ms … 64.832 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.724 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.131 ms ±  1.811 ms  ┊ GC (mean ± σ):  3.07% ± 6.50%

  ▆█▅▄▄▄▃▂▁▃▄▃▂▁▁▁                                            
  █████████████████▇█▇█▆▇▇▇▆▇█▇▇▇▆▆▆▆▇▅▄▅▄▅▅▆▅▄▄▆▆▄▆▄▄▁▅▄▄▅▄ █
  2.49 ms      Histogram: log(frequency) by time     6.57 ms <

 Memory estimate: 7.65 MiB, allocs estimate: 6.

In [70]:
# solve using Woodbury formula
@benchmark $W \ reshape($b, n, 1) # hack; need to file an issue 

BenchmarkTools.Trial: 10000 samples with 8 evaluations per sample.
 Range (min … max):  3.302 μs … 196.781 μs  ┊ GC (min … max):  0.00% … 95.91%
 Time  (median):     3.870 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   4.780 μs ±  10.525 μs  ┊ GC (mean ± σ):  13.83% ±  6.15%

    ▂▆█▃                                                       
  ▂▆████▇▅▄▅▅▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  3.3 μs          Histogram: frequency by time        7.43 μs <

 Memory estimate: 32.66 KiB, allocs estimate: 18.

### Matrix-vector multiplication

In [71]:
# multiplication without using Woodbury structure
@benchmark $Wfull * $b

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   83.584 μs …   2.941 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     136.750 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   176.158 μs ± 117.870 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

   █▁▅▆                                                          
  ▇████▇▅▅▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▁▂▂▂ ▃
  83.6 μs          Histogram: frequency by time          636 μs <

 Memory estimate: 8.06 KiB, allocs estimate: 3.

In [72]:
# multiplication using Woodbury structure
@benchmark $W * $b

BenchmarkTools.Trial: 10000 samples with 9 evaluations per sample.
 Range (min … max):  2.398 μs … 175.745 μs  ┊ GC (min … max):  0.00% … 94.73%
 Time  (median):     2.648 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   3.226 μs ±   7.616 μs  ┊ GC (mean ± σ):  14.79% ±  6.16%

    ▂▄▃▄█▆▄▃                                                   
  ▅█████████▇▅▄▄▃▃▂▂▂▂▂▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  2.4 μs          Histogram: frequency by time        4.39 μs <

 Memory estimate: 32.44 KiB, allocs estimate: 16.

### Determinant

In [73]:
# determinant without using Woodbury structure
@benchmark det($Wfull)

BenchmarkTools.Trial: 1048 samples with 1 evaluation per sample.
 Range (min … max):  3.572 ms … 17.036 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.203 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.771 ms ±  1.471 ms  ┊ GC (mean ± σ):  2.19% ± 4.89%

   █                                                          
  ▇██▇▆▅▄▄▅▄▄▄▃▃▃▃▃▃▃▂▃▃▃▃▃▂▃▃▃▃▃▃▂▂▂▃▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂ ▃
  3.57 ms        Histogram: frequency by time        9.24 ms <

 Memory estimate: 7.65 MiB, allocs estimate: 6.

In [74]:
# determinant using Woodbury structure
@benchmark det($W)

BenchmarkTools.Trial: 10000 samples with 250 evaluations per sample.
 Range (min … max):  304.000 ns … 96.323 μs  ┊ GC (min … max): 0.00% … 99.55%
 Time  (median):     334.164 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   360.498 ns ±  1.173 μs  ┊ GC (mean ± σ):  4.52% ±  1.41%

        ▃██▄ ▁▁                                                 
  ▂▃▃▃▃▅████████▇▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  304 ns          Histogram: frequency by time          462 ns <

 Memory estimate: 368 bytes, allocs estimate: 4.

## Easy plus border

Easy plus border: For $\mathbf{A}$ pd and $\mathbf{V}$ full row rank,
$$
	\begin{pmatrix}
	\mathbf{A} & \mathbf{V}^T \\
	\mathbf{V} & \mathbf{0}
	\end{pmatrix}^{-1} = \begin{pmatrix}
	\mathbf{A}^{-1} - \mathbf{A}^{-1} \mathbf{V}^T (\mathbf{V} \mathbf{A}^{-1} \mathbf{V}^T)^{-1} \mathbf{V} \mathbf{A}^{-1} & \mathbf{A}^{-1} \mathbf{V}^T (\mathbf{V} \mathbf{A}^{-1} \mathbf{V}^T)^{-1} \\
	(\mathbf{V} \mathbf{A}^{-1} \mathbf{V}^T)^{-1} \mathbf{V} \mathbf{A}^{-1} & - (\mathbf{V} \mathbf{A}^{-1} \mathbf{V}^T)^{-1}
	\end{pmatrix}.
$$
**Anyone interested writing a package?**

## Orthogonal matrix

Orthogonal $\mathbf{A}$: $n^2$ flops **at most**. Why? Permutation matrix, Householder matrix, Jacobi matrix, ... take less.

## Toeplitz matrix

Toeplitz systems (constant diagonals):
$$
	\mathbf{T} = \begin{pmatrix}
	r_0 & r_1 & r_2 & r_3 \\
	r_{-1} & r_0 & r_1 & r_2 \\
	r_{-2} & r_{-1} & r_0 & r_1 \\
	r_{-3} & r_{-2} & r_{-1} & r_0
	\end{pmatrix}.
$$
$\mathbf{T} \mathbf{x} = \mathbf{b}$, where $\mathbf{T}$ is pd and Toeplitz, can be solved in $O(n^2)$ flops. Durbin algorithm (Yule-Walker equation), Levinson algorithm (general $\mathbf{b}$), Trench algorithm (inverse). These matrices occur in auto-regressive models and econometrics.

* [`ToeplitzMatrices.jl`](https://github.com/JuliaMatrices/ToeplitzMatrices.jl) package can be useful.

## Circulant matrix

Circulant systems: Toeplitz matrix with wraparound
$$
	C(\mathbf{z}) = \begin{pmatrix}
	z_0 & z_4 & z_3 & z_2 & z_1 \\
	z_1 & z_0 & z_4 & z_3 & z_2 \\
	z_2 & z_1 & z_0 & z_4 & z_3 \\
	z_3 & z_2 & z_1 & z_0 & z_4 \\
	z_4 & z_3 & z_2 & z_1 & z_0
	\end{pmatrix},
$$
FFT type algorithms: DCT (discrete cosine transform) and DST (discrete sine transform).

## Vandermonde matrix

Vandermonde matrix: such as in interpolation and approximation problems
$$
	\mathbf{V}(x_0,\ldots,x_n) = \begin{pmatrix}
	1 & 1 & \cdots & 1 \\
	x_0 & x_1 & \cdots & x_n \\
	\vdots & \vdots & & \vdots \\
	x_0^n & x_1^n & \cdots & x_n^n
	\end{pmatrix}.
$$
$\mathbf{V} \mathbf{x} = \mathbf{b}$ or $\mathbf{V}^T \mathbf{x} = \mathbf{b}$ can be solved in $O(n^2)$ flops.

## Cauchy-like matrix

Cauchy-like matrices:
$$
	\Omega \mathbf{A} - \mathbf{A} \Lambda = \mathbf{R} \mathbf{S}^T,
$$
where $\Omega = \text{diag}(\omega_1,\ldots,\omega_n)$ and $\Lambda = \text{diag}(\lambda_1,\ldots, \lambda_n)$. $O(n)$ flops for LU and QR.

## Structured-rank matrix

Structured-rank problems: semiseparable matrices (LU and QR takes $O(n)$ flops), quasiseparable matrices, ...

# LinearSolve.jl package

[LinearSolve.jl](https://github.com/SciML/LinearSolve.jl) is meta-package in Julia that defines a unified interface for solving linear equations and makes it easy switching linear solvers while maintaining the maximum efficiency.

TODO: examples.